# 04. 유전체 특징 산출: APOE, 형질군별 희소 PRS, 유전적 주성분

연구기획서 3.3절의 G 블록을 만든다. 수령 자료의 유전체는 실제 rsID를 가진 lead 변이 178개와 합성 ID의 LD 이웃, 배경 null SNP로 구성되어 있으므로(노트북 02), 전장 PRS 대신 **lead 변이 기반 희소 PRS**를 쓴다.

1. APOE: rs429358과 rs7412의 대립유전자 방향을 bim에서 확인해 ε4 보유 수와 ε2 보유 수를 만든다.
2. 형질군 분류: lead 변이를 문헌(GWAS Catalog 및 원 논문 기준 잠정 분류)으로 먼저 묶고, 문헌에 없는 변이는 **개발군에서만** 관찰된 형질 연관(가장 강한 형질군, Bonferroni 기준)으로 배정한다. 두 출처를 모두 기록한다.
3. 희소 PRS: 형질군별로 개발군에서 규제 회귀로 학습한 가중치를 전체에 적용한다(평가군은 적용만). 가중치 학습의 표적은 질환과 대사 지표이지 뇌 체적이 아니다.
4. 유전적 주성분: 배경 SNP 5,000개를 무작위 추출해 PCA(개발군에서 적합, 전체에 투영).
5. `genomic_features.csv`(T_ID + 특징)와 `lead_annotation.csv`(변이별 형질군, 출처, 가중치)를 저장한다.

출력 셀에는 집계값만 남긴다.

In [1]:
import json, re, warnings
from pathlib import Path
import numpy as np, pandas as pd
from scipy import stats
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.decomposition import PCA
from sklearn.metrics import roc_auc_score
warnings.filterwarnings("ignore")
SEED = 20260915
ROOT = Path.cwd()
while not (ROOT / "02_Data").exists() and ROOT != ROOT.parent: ROOT = ROOT.parent
G = ROOT / "02_Data" / "유전체_합성데이터"; DERIVED = ROOT / "05_분석결과" / "derived"; TAB = ROOT / "05_분석결과" / "tables"
B = pd.read_csv(DERIVED / "base_analysis.csv").set_index("T_ID")
D = pd.read_csv(DERIVED / "lead_dosage.csv").set_index("T_ID")      # 노트북 02 산출: 10,000 × 178 (A1 = 효과 = minor 대립유전자 개수)
D = D.loc[B.index]
dev = (B.split == "dev").values
print("대상자", len(B), "| 개발군", int(dev.sum()), "| lead 변이", D.shape[1])
fam = pd.read_csv(G / "synthetic_kchip_chr1.fam", sep=r"\s+", header=None, names=["FID", "IID", "PAT", "MAT", "SEX", "PHENO"])
N = len(fam); bps = (N + 3) // 4
bims = pd.concat([pd.read_csv(G / f"synthetic_kchip_chr{c}.bim", sep=r"\s+", header=None, names=["CHR", "SNP", "CM", "BP", "A1", "A2"]).assign(idx=lambda d: np.arange(len(d))) for c in range(1, 23)], ignore_index=True)
lead_bim = bims[bims.SNP.isin(D.columns)].set_index("SNP")

대상자 10000 | 개발군 8000 | lead 변이 178


## 1. APOE ε4, ε2

실제 유전체라면 ε4는 rs429358의 C 대립유전자, ε2는 rs7412의 T 대립유전자다. 그러나 합성 bim의 대립유전자 문자(rs429358 A/C, rs7412 T/A)는 실제 염기와 다르게 임의로 부여되어 있어 문자로 방향을 정할 수 없다. 유전체 코드북이 A1 = 효과 = minor 대립유전자로 정의하고, 두 변이의 minor 빈도(각 0.09)가 한국인 ε4와 ε2 빈도와 맞으므로, **ε4 = rs429358의 A1 개수, ε2 = rs7412의 A1 개수**로 정의한다(dosage 그대로). 위상 정보가 없으므로 두 변이가 모두 이형인 경우는 ε2/ε4로 간주하고 수를 기록한다.

In [2]:
print("bim 대립유전자(합성, 임의 문자): rs429358 A1/A2 =", lead_bim.loc["rs429358", ["A1", "A2"]].tolist(), "| rs7412 A1/A2 =", lead_bim.loc["rs7412", ["A1", "A2"]].tolist())
print("minor 빈도: rs429358 %.3f, rs7412 %.3f" % (D["rs429358"].mean() / 2, D["rs7412"].mean() / 2))
e4 = D["rs429358"].values.copy(); e2 = D["rs7412"].values.copy()   # A1(효과 = minor) 개수
feat = pd.DataFrame(index=B.index)
feat["apoe_e4_count"], feat["apoe_e2_count"] = e4, e2
feat["apoe_e4_carrier"] = (e4 >= 1).astype(int)
amb = int(((e4 == 1) & (e2 == 1)).sum())
print("ε4 보유 수 분포:", dict(pd.Series(e4).value_counts().sort_index()), "| ε4 보유자 %.1f%%" % (100 * feat.apoe_e4_carrier.mean()))
print("ε2 보유 수 분포:", dict(pd.Series(e2).value_counts().sort_index()), "| 두 변이 모두 이형(위상 불명) n =", amb)

bim 대립유전자(합성, 임의 문자): rs429358 A1/A2 = ['A', 'C'] | rs7412 A1/A2 = ['T', 'A']
minor 빈도: rs429358 0.091, rs7412 0.089
ε4 보유 수 분포: {0.0: np.int64(8268), 1.0: np.int64(1651), 2.0: np.int64(81)} | ε4 보유자 17.3%
ε2 보유 수 분포: {0.0: np.int64(8295), 1.0: np.int64(1627), 2.0: np.int64(78)} | 두 변이 모두 이형(위상 불명) n = 284


## 2. lead 변이의 형질군 분류

문헌 분류(잠정, 제출 전 GWAS Catalog 대조 필요)를 먼저 적용하고, 나머지는 개발군에서의 형질 연관으로 배정한다. 형질군: T2D(당뇨 및 혈당), HTN(혈압), LIPID(지질), OBESITY(비만), ALCOHOL(음주 대사), APOE(APOE 및 AD 관련), OTHER(뇌 및 기타).

In [3]:
LIT = {
 "T2D": ["rs7903146","rs1801282","rs5219","rs10811661","rs13266634","rs1111875","rs10830963","rs4402960","rs7754840","rs10946398","rs2237892","rs163184",
         "rs4607103","rs7578597","rs10923931","rs340874","rs2233580","rs7172432","rs1552224","rs896854","rs972283","rs4457053","rs7961581","rs1531343",
         "rs2191349","rs12779790","rs10010131","rs11708067","rs7957197","rs4430796","rs7034200","rs16861329","rs10423928"],
 "HTN": ["rs1799998","rs4961","rs699","rs5443","rs1799983","rs1801253","rs17367504","rs880315","rs3754777","rs2932538","rs3774372","rs16998073","rs1173771",
         "rs11014166","rs1530440","rs932764","rs11191548","rs381815","rs17249754","rs3184504","rs1378942","rs2521501","rs12946454","rs16948048","rs2384550",
         "rs13139571","rs5068","rs4656461","rs1192415","rs7865618","rs11066280"],
 "LIPID": ["rs2075291","rs662799","rs651821","rs328","rs1800588","rs3764261","rs708272","rs12916","rs629301","rs11206510","rs6511720","rs693","rs4299376",
           "rs2954029","rs174547","rs6065906","rs1260326","rs16996148","rs17145738","rs4846914","rs2131925","rs6882076","rs255052","rs2338104","rs11023332","rs12229654"],
 "OBESITY": ["rs8050136","rs6567160","rs543874"],
 "ALCOHOL": ["rs671","rs1229984"],
 "APOE": ["rs429358","rs7412","rs2075650","rs405509","rs769446","rs449647"],
 "OTHER": ["rs6265","rs738409","rs1065852","rs1801260","rs112735431","rs10246939","rs713598","rs1726866","rs4236601","rs1063192","rs2157719","rs10757272","rs944289","rs1183910"],
}
lit_map = {snp: g for g, l in LIT.items() for snp in l if snp in D.columns}
# 개발군 형질 연관으로 나머지 배정 (형질군 대표 변수: T2D=dm,glu / HTN=htn,sbp / LIPID=lip,ldl,tg,hdl / OBESITY=bmi)
fam_traits = {"T2D": ["dm", "glu", "hba1c"], "HTN": ["htn", "sbp", "dbp"], "LIPID": ["lip", "ldl", "tg", "hdl", "tc"], "OBESITY": ["bmi", "waist"]}
Bd = B[dev]; Dd = D[dev]
rows = []
for snp in D.columns:
    r = {"snp": snp, "chr": int(lead_bim.loc[snp, "CHR"]), "maf": round(D[snp].mean() / 2, 3), "lit_group": lit_map.get(snp, "")}
    best_g, best_p = None, 1.0
    for g, ts in fam_traits.items():
        for t in ts:
            ok = Bd[t].notna().values
            p = stats.linregress(Dd[snp].values[ok], Bd[t].values[ok]).pvalue
            r[f"p_{t}"] = p
            if p < best_p: best_g, best_p = g, p
    thr = 0.05 / len(D.columns)
    r["data_group"], r["data_p"] = (best_g if best_p < thr else "NONE"), best_p
    r["data_groups_all"] = ";".join(g for g, ts in fam_traits.items() if min(r[f"p_{t}"] for t in ts) < thr)   # 다면발현 허용
    r["group"] = r["lit_group"] if r["lit_group"] else (r["data_group"] if r["data_group"] != "NONE" else "OTHER")
    r["source"] = "문헌" if r["lit_group"] else ("자료 연관" if r["data_group"] != "NONE" else "미배정(OTHER)")
    rows.append(r)
ann = pd.DataFrame(rows)
# PRS용 형질군 집합 = 문헌 배정 ∪ 개발군 자료 연관(다면발현 포함)
prs_sets = {g: sorted(set(ann.loc[ann.lit_group == g, "snp"]) | set(ann.loc[ann.data_groups_all.str.contains(g), "snp"])) for g in fam_traits}
print("PRS용 형질군 집합 크기(문헌 ∪ 자료 연관):", {g: len(v) for g, v in prs_sets.items()})
print("형질군별 lead 수:", ann.group.value_counts().to_dict())
print("출처별:", ann.source.value_counts().to_dict())
agree = ann[(ann.lit_group != "") & (ann.data_group != "NONE")]
print("문헌과 자료 연관이 모두 있는 %d개 중 일치 %d개 (%.0f%%)" % (len(agree), (agree.lit_group == agree.data_group).sum(), 100 * (agree.lit_group == agree.data_group).mean()))
print("불일치 예:", agree[agree.lit_group != agree.data_group][["snp", "lit_group", "data_group"]].head(8).to_string(index=False))

PRS용 형질군 집합 크기(문헌 ∪ 자료 연관): {'T2D': 36, 'HTN': 34, 'LIPID': 32, 'OBESITY': 5}
형질군별 lead 수: {'OTHER': 75, 'T2D': 34, 'HTN': 31, 'LIPID': 27, 'APOE': 6, 'OBESITY': 3, 'ALCOHOL': 2}
출처별: {'문헌': 115, '미배정(OTHER)': 61, '자료 연관': 2}
문헌과 자료 연관이 모두 있는 63개 중 일치 53개 (84%)
불일치 예:        snp lit_group data_group
 rs1260326     LIPID        T2D
rs10757272     OTHER        HTN
 rs2157719     OTHER        HTN
     rs671   ALCOHOL        HTN
 rs8050136   OBESITY        T2D
  rs449647      APOE      LIPID
  rs405509      APOE      LIPID
  rs769446      APOE      LIPID


## 3. 형질군별 희소 PRS

각 형질군의 SNP dosage를 입력, 형질군의 대표 질환(당뇨, 고혈압, 고지혈증) 또는 대사 지표(BMI)를 표적으로 **개발군에서만** 규제 회귀를 적합해 가중치를 얻고, 전체 대상자에 적용한 뒤 개발군 평균과 SD로 표준화한다. 음주 대사군은 문헌 방향(rs671 A, rs1229984 A 보호 대립유전자)의 단순 합으로 만든다. 평가군의 형질 AUROC는 가중치가 과적합되지 않았는지 확인하는 용도다.

In [4]:
ev = ~dev
prs = pd.DataFrame(index=B.index); weights = {}
def fit_prs(name, snps, target, binary):
    X = D[snps].values; Xd, yd = X[dev], B[target].values[dev]
    ok = ~np.isnan(yd)
    if binary:
        m = LogisticRegression(C=0.05, max_iter=5000).fit(Xd[ok], yd[ok].astype(int)); w = m.coef_.ravel()
    else:
        m = Ridge(alpha=50.0).fit(Xd[ok], yd[ok]); w = m.coef_.ravel()
    s = X @ w; mu, sd = s[dev].mean(), s[dev].std()
    prs[name] = (s - mu) / sd; weights[name] = dict(zip(snps, np.round(w, 5)))
    ye = B[target].values[ev]; oke = ~np.isnan(ye)
    perf = roc_auc_score(ye[oke].astype(int), prs[name].values[ev][oke]) if binary else np.corrcoef(ye[oke], prs[name].values[ev][oke])[0, 1]
    print(f"  {name:<12} SNP {len(snps):>3}개 → 표적 {target:<5} 평가군 {'AUROC' if binary else '상관'} = {perf:.3f}")
grp = lambda g: ann.loc[ann.group == g, "snp"].tolist()
fit_prs("prs_t2d", prs_sets["T2D"], "dm", True)
fit_prs("prs_htn", prs_sets["HTN"], "htn", True)
fit_prs("prs_lipid", prs_sets["LIPID"], "lip", True)
fit_prs("prs_obesity", prs_sets["OBESITY"], "bmi", False)
alc = [s for s in ["rs671", "rs1229984"] if s in D.columns]
prs["prs_alcohol"] = D[alc].sum(axis=1).values; prs["prs_alcohol"] = (prs.prs_alcohol - prs.prs_alcohol[dev].mean()) / prs.prs_alcohol[dev].std()
print("  prs_alcohol   SNP %d개 (단순 합)" % len(alc))

  prs_t2d      SNP  36개 → 표적 dm    평가군 AUROC = 0.821
  prs_htn      SNP  34개 → 표적 htn   평가군 AUROC = 0.812
  prs_lipid    SNP  32개 → 표적 lip   평가군 AUROC = 0.845
  prs_obesity  SNP   5개 → 표적 bmi   평가군 상관 = 0.034
  prs_alcohol   SNP 2개 (단순 합)


## 4. 유전적 주성분 (배경 SNP 5,000개 표본, 개발군 적합)

In [5]:
LUT = np.array([2, np.nan, 1, 0], dtype=float)
rng = np.random.default_rng(SEED)
bg = bims[bims.SNP.str.startswith("bg_")]
pick = bg.sample(5000, random_state=SEED).sort_values(["CHR", "idx"])
def read_snps(chrom, idxs):
    out = []
    with open(G / f"synthetic_kchip_chr{chrom}.bed", "rb") as f:
        for i in idxs:
            f.seek(3 + i * bps); buf = np.frombuffer(f.read(bps), dtype=np.uint8)
            bits = np.unpackbits(buf, bitorder="little").reshape(-1, 2); out.append(LUT[bits[:, 0] + 2 * bits[:, 1]][:N])
    return np.array(out, dtype=np.float32)
mats = [read_snps(c, g.idx.tolist()) for c, g in pick.groupby("CHR")]
X = np.vstack(mats).T                                   # N × 5000, fam 순서
X = pd.DataFrame(X, index=fam.IID.values).loc[B.index].values
X = np.where(np.isnan(X), np.nanmean(X, axis=0), X)
mu, sd = X[dev].mean(0), X[dev].std(0) + 1e-9
Xs = (X - mu) / sd
pca = PCA(n_components=10, random_state=SEED).fit(Xs[dev])
PC = pca.transform(Xs)
for k in range(10): prs[f"pc{k+1}"] = PC[:, k]
print("PC 설명 분산(%):", [round(float(v) * 100, 3) for v in pca.explained_variance_ratio_])
print("(집단 구조가 없는 합성 자료라면 각 PC가 0.1% 안팎으로 고르게 나온다)")

PC 설명 분산(%): [0.061, 0.06, 0.06, 0.059, 0.059, 0.059, 0.059, 0.059, 0.058, 0.058]
(집단 구조가 없는 합성 자료라면 각 PC가 0.1% 안팎으로 고르게 나온다)


## 5. 저장과 점검

In [6]:
feat = feat.join(prs)
feat.reset_index().to_csv(DERIVED / "genomic_features.csv", index=False)
ann.to_csv(DERIVED / "lead_annotation.csv", index=False, encoding="utf-8-sig")
json.dump({"weights": weights, "annotation_groups": {g: grp(g) for g in ["T2D", "HTN", "LIPID", "OBESITY", "ALCOHOL", "APOE", "OTHER"]}, "prs_sets": prs_sets,
           "apoe_rule": "합성 bim의 대립유전자 문자는 임의이므로 e4 = rs429358 A1(minor) 개수, e2 = rs7412 A1(minor) 개수", "pca": {"n_background_snps": 5000, "seed": SEED}},
          open(DERIVED / "genomic_features_spec.json", "w", encoding="utf-8"), ensure_ascii=False, indent=2)
print("G 블록 특징:", list(feat.columns))
print("\n개발군에서 각 특징과 해마 z의 상관(참고; 모델 비교의 판단 근거로 쓰지 않음):")
print(pd.concat([feat, B.z_hip], axis=1)[dev].corr()["z_hip"].drop("z_hip").round(3).to_string())
print("\n저장:", (DERIVED / "genomic_features.csv").name, (DERIVED / "lead_annotation.csv").name)

G 블록 특징: ['apoe_e4_count', 'apoe_e2_count', 'apoe_e4_carrier', 'prs_t2d', 'prs_htn', 'prs_lipid', 'prs_obesity', 'prs_alcohol', 'pc1', 'pc2', 'pc3', 'pc4', 'pc5', 'pc6', 'pc7', 'pc8', 'pc9', 'pc10']

개발군에서 각 특징과 해마 z의 상관(참고; 모델 비교의 판단 근거로 쓰지 않음):
apoe_e4_count     -0.006
apoe_e2_count      0.002
apoe_e4_carrier   -0.004
prs_t2d           -0.023
prs_htn           -0.097
prs_lipid         -0.024
prs_obesity       -0.036
prs_alcohol       -0.022
pc1               -0.007
pc2               -0.020
pc3               -0.010
pc4                0.003
pc5                0.003
pc6                0.000
pc7               -0.000
pc8               -0.005
pc9               -0.007
pc10               0.015

저장: genomic_features.csv lead_annotation.csv
